In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU runtime before running AST fine-tuning.")


CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
!rm -rf /content/music-genre-classification
!cp -r "/content/drive/MyDrive/music-genre-classification" /content/music-genre-classification
%cd /content/music-genre-classification

!pip install -q -r requirements.txt


/content/music-genre-classification


In [4]:
from pathlib import Path

required = [
    Path("features/mel_specs.npz"),
    Path("features/ast_inputs.npz"),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing cached fine-tuning inputs: " + ", ".join(missing))

print("Cached fine-tuning inputs ready:", ", ".join(str(path) for path in required))


Cached fine-tuning inputs ready: features/mel_specs.npz, features/ast_inputs.npz


In [ ]:
from pathlib import Path
import shutil
import subprocess

run_dir = Path("results/5.4 Fine Tuning")
if run_dir.exists():
    shutil.rmtree(run_dir)

!python3 transfer_learning_fine_tuning.py

drive_root = Path("/content/drive/MyDrive/music-genre-classification")
drive_results = drive_root / "results"
drive_features = drive_root / "features"
drive_results.mkdir(parents=True, exist_ok=True)
drive_features.mkdir(parents=True, exist_ok=True)

src = Path("results/5.4 Fine Tuning")
dst = drive_results / "5.4 Fine Tuning"
if src.exists():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print("Copied", src, "->", dst)

ast_input_cache = Path("features/ast_inputs.npz")
if ast_input_cache.exists():
    shutil.copy2(ast_input_cache, drive_features / ast_input_cache.name)
    print("Copied", ast_input_cache, "->", drive_features / ast_input_cache.name)

for filename in ["model_comparison.csv", "model_comparison.png"]:
    src = Path("results") / filename
    if src.exists():
        shutil.copy2(src, drive_results / filename)
        print("Copied", src, "->", drive_results / filename)


preprocessor_config.json: 100% 297/297 [00:00<00:00, 1.52MB/s]

[Load cached AST inputs]
Cache: features/ast_inputs.npz
config.json: 100% 26.8k/26.8k [00:00<00:00, 51.6MB/s]
[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `527`.
model.safetensors: 100% 346M/346M [00:03<00:00, 90.9MB/s]
Loading weights: 100% 203/203 [00:00<00:00, 12282.80it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                         
------------------------+----------+-----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([8])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([8, 768])

Note

In [ ]:
import pandas as pd
from pathlib import Path

path = Path("results/5.4 Fine Tuning/metrics.csv")
if not path.exists():
    raise FileNotFoundError("Missing results/5.4 Fine Tuning/metrics.csv. Run fine-tuning first.")
metrics = pd.read_csv(path)
metrics.sort_values(["split", "f1_macro"], ascending=[True, False])


In [ ]:
from google.colab import runtime

runtime.unassign()
